In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import faiss
from dotenv import load_dotenv
import os
from google import genai

---

In [ ]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
EMBED_MODEL = os.getenv("EMBED_MODEL")
GEN_MODEL = os.getenv("GEN_MODEL")
OUTPUT_DIMENSION = os.getenv("OUTPUT_DIMENSION")

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
index = faiss.read_index("../output_index/text-embed-04")

---

# Process

In [ ]:
movie_df = pd.read_csv("../data/movie_infos.csv")
movie_df["vectorID"] = movie_df.index.values
movie_df

In [ ]:
ratings = pd.read_csv("../data/rating.csv")

In [ ]:
test = movie_df.merge(ratings.groupby(by="movieId").count()
                      ["userId"], on="movieId", how="left")
test.fillna(0)

C = test["rating"].mean()
m = 1300


def weight_rating(row):

    v = row["userId"]
    R = row["rating"]

    return (v/(v+m)) * R + (m/(v+m)) * C


movie_df["weight_rating"] = test.apply(weight_rating, axis=1)


def convert_prompt(row):
    title = row["title"]
    genres = row["genres"]
    tags = row["tags"]
    # link = f"https://www.imdb.com/title/{row["imdbId"]}/"
    # rating = round(row["weight_rating"],4)

    return f"Movie's title: {title}\ngenres: {genres}\ntags: {tags}"


movie_df["page_content"] = movie_df.apply(convert_prompt, axis=1)
movie_df.set_index("movieId", inplace=True)

In [ ]:
movie_df = movie_df[["vectorID", "title", "genres", "tags",
                     "weight_rating", "imdbId", "page_content"]]

In [ ]:
# movie_df.to_csv("../data/movie_infos_2.csv")

---

# Data

In [ ]:
movie_df = pd.read_csv("../data/movie_infos_2.csv")
movie_df.set_index("movieId", inplace=True)

In [ ]:
movie_df

In [ ]:
user_rates = pd.read_pickle("../data/sorted_ratings.pkl")
user_rates

---

# Luồng chạy

### Lấy ra tất cả các phim user bất kì đã xem

In [ ]:
movie_list = user_rates.loc[1].merge(movie_df, on="movieId")[
    ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]

In [ ]:
movie_list

### Hàm recommend top k phim giống với một phim nhất

In [ ]:
def get_top_k(movie, k=5):
    embed_input = index.reconstruct(movie["vectorID"])
    D, I = index.search(np.array([embed_input]), k)

    return D, I

### Tính điểm ranking của các phim được recommend từ mỗi phim mà user đã xem

In [ ]:
recommend_dict = {}
negative_threshold = 3
negative_alpha = -0.5


for movieId, row in movie_list.iterrows():

    m_id = movieId

    # m_content = row["page_content"]

    m_rating = row["rating"]

    m_weight_rating = row["weight_rating"]

    combine_rating = 0.8 * m_rating + 0.2 * m_weight_rating

    D, I = get_top_k(row, k=5)

    rcm_ids = I.flatten().tolist()[1:]

    rcm_dists = D.flatten().tolist()[1:]

    for id_, dist_ in zip(rcm_ids, rcm_dists):
        if (m_rating < negative_threshold):
            wgt = (combine_rating + negative_alpha *
                   (negative_threshold - m_rating)) * dist_
        else:

            wgt = combine_rating * dist_

        recommend_dict[id_] = recommend_dict.get(id_, 0) + wgt

### Lấy ra top K recommend cuối cùng

In [ ]:
K = 10
sorted_candidates = sorted(recommend_dict.items(),
                           key=lambda x: x[1], reverse=True)
final_rcm_list = [id_ for id_, _ in sorted_candidates[:K]]
final_rcm_list

In [ ]:
movie_df.iloc[final_rcm_list]

### LLM để phân phim thành nhiều thể loại

In [ ]:
recommend_contents = "\n======\n".join(
    movie_df.iloc[final_rcm_list]["page_content"].tolist())

In [ ]:
system_prompt = (
    "You are an AI assistant majoring in categorize things. Your task is to group the given movies by their genres into categories"
    """Your task is as follows:
        1. Group the movies into categories based on their genres.
        2. If a movie has multiple genres, assign it to the category corresponding to its primary genre (choose the first listed genre).
        3. For each category, list all movies that belong to that category."""
)

user_prompt = f"""
Below is a list of movies. Each movie is formatted as follows:

--------------------------------------------------
Movie Format:
(Title)
(Genres)        // A comma-separated list of genres.
(Tags)          // A comma-separated list of tags.
--------------------------------------------------
For example:
Now, Voyager (1942)
Drama, Romance
Classic, Timeless, Iconic
--------------------------------------------------

Please group these movies into categories based on their genres in the following format:
**Genre1**
- Movie1
- Movie2
...

**Genre2**
- Movie1
...

Here are the movies:
{recommend_contents}
"""

In [ ]:
res = client.models.generate_content(
    model=GEN_MODEL,
    contents=[system_prompt, user_prompt]
)

In [ ]:
print(res.text)

In [ ]:
system_prompt_2 = "You are a JSON formatter AI assistant, your job is convert given data into valid JSON format."
user_prompt_2 = f"""Please convert the following text into JSON format as follow:  
{{
  "Genre1": [
    {{"title": "Movie Title 1"}},
    {{"title": "Movie Title 3"}}
  ],
  "Genre2": [
    {{"title": "Movie Title 1"}},
    {{"title": "Movie Title 5"}},
    ...
  ],
  ...
}}


Here's the text:
{res.text}

Please return ONLY the JSON in the format above, WITHOUT any explaination or infos.

"""

In [ ]:
res2 = client.models.generate_content(
    model=GEN_MODEL,
    contents=[system_prompt_2, user_prompt_2]
)

In [ ]:
print(res2.text)

---

# Đánh giá

In [ ]:
from sklearn.model_selection import train_test_split
import time

---

### Cách 1: Sử dụng LLM để embed thông tin của phim (Content based)

In [ ]:
def get_rcm_dict(movies, k=5, negative_alpha=-0.5, negative_threshold=3):
    user_dislike = []
    user_like = []

    recommend_dict = {}

    for movieId, row in movies.iterrows():

        m_id = movieId

        m_content = row["page_content"]


        m_rating = row["rating"]

        m_weight_rating = row["weight_rating"]


        combine_rating = 0.8 * m_rating + 0.2 * m_weight_rating


        D, I = get_top_k(row, k=k)


        rcm_ids = I.flatten().tolist()[1:]

        rcm_dists = D.flatten().tolist()[1:]


        for id_, dist_ in zip(rcm_ids, rcm_dists):


            if (m_rating < negative_threshold):
                if (id_ in user_like):
                    continue
                user_dislike.append(row["vectorID"])

                wgt = (combine_rating + negative_alpha * (5 - m_rating)) * dist_

                recommend_dict[id_] = recommend_dict.get(id_, 0) - wgt


            else:
                user_like.append(row["vectorID"])

                wgt = combine_rating * dist_


                recommend_dict[id_] = recommend_dict.get(id_, 0) + wgt


        for id_ in set(user_dislike):
            recommend_dict.pop(id_, -1)

    return recommend_dict

In [ ]:
user_size = user_rates.index.get_level_values(0).unique().shape[0]
K = [10, 20, 50]


metrics_results = {
    "HR": {k: [] for k in K},
    "Precision": {k: [] for k in K},
    "Recall": {k: [] for k in K},
    "NDCG": {k: [] for k in K},
}


counter = 0


for i in tqdm(range(1, user_size + 1)):

    precision = []
    recall_list = []
    ndcg_list = []

    if (counter == 5000):
        break
    user_ = user_rates.loc[i]

    if (user_.shape[0] < 100):
        continue

    if (counter % 100 == 0):
        print(counter)

    counter += 1

    user_ = user_.merge(movie_df, on="movieId")[
        ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]

    train, test = train_test_split(user_, test_size=0.25, shuffle=False)

    recommend_dict = get_rcm_dict(train, k=10)

    for k_ in K:
        sorted_candidates = sorted(recommend_dict.items(),
                                   key=lambda x: x[1], reverse=True)

        final_rcm_list = movie_df.iloc[[id_ for id_,
                                        _ in sorted_candidates[:k_]]]

        overlap = np.intersect1d(final_rcm_list.index, test.index.values).size

        hr_k = 1 if overlap > 0 else 0

        precisionk = (overlap / k_)
        precision.append(precisionk)

        recall_k = overlap / len(test.index.values)
        recall_list.append(recall_k)

        test_movies = test.index.values

        relevances = [
            1 if movie in test_movies else 0 for movie in final_rcm_list]

        dcg = np.sum([rel / np.log2(idx + 2)
                      for idx, rel in enumerate(relevances)])

        ideal_hits = min(len(test_movies), k_)

        idcg = np.sum([1 / np.log2(i + 2) for i in range(ideal_hits)])

        ndcg_k = dcg / idcg if idcg > 0 else 0
        ndcg_list.append(ndcg_k)

        metrics_results["Precision"][k_].append(precisionk)

        metrics_results["Recall"][k_].append(recall_k)

        metrics_results["NDCG"][k_].append(ndcg_k)

        metrics_results["HR"][k_].append(hr_k)

In [ ]:
final_results = {metric: [np.mean(metrics_results[metric][k])
                          for k in K] for metric in metrics_results}
results_df = pd.DataFrame(final_results, index=K).T

results_df

---

### Cách 2: Sử dụng LLM để tạo ra vector phim đặc trưng của user

In [ ]:
movie_list = user_rates.loc[2].merge(movie_df, on="movieId")[
    ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]


def convert_prompt_2(row):
    rating = row["rating"]
    # link = f"https://www.imdb.com/title/{row["imdbId"]}/"
    # rating = round(row["weight_rating"],4)

    return f"{row["page_content"]} | rating: {rating}"


movie_list["page_content"] = movie_list.apply(convert_prompt_2, axis=1)

In [ ]:
def get_user_desc(contents):
    sys_prompt = """
    You are a recommendation system assistant. Your task is to analyze a user's watched movies and generate a concise feature vector representing their preferences. The feature vector must include relevant genres and descriptive tags.

    Extract common genres across the watched movies.
    Identify key descriptive tags based on themes, moods, and characteristics.
    If a movie has rating < 3.0 that means that user do not like that movie
    Format the output strictly as: 'genres: ... | tags: ...'.
    Do not include explanations, extra text, or formatting variations.
"""

    user_prompt = f"""
    Extract a feature vector from the user's watched movies. Use only genres and tags in the output. Format: 'genres: ... | tags: ...'. No extra text & strip all the unneccessary space.
    Here's the user watched movies:
    {contents}
"""

    res = client.models.generate_content(
        model=GEN_MODEL,
        contents=[sys_prompt, user_prompt]
    )

    return res


def _get_embed(feature_movies):
    res = client.models.embed_content(
        model=EMBED_MODEL,
        contents=feature_movies,
        config={
            "output_dimensionality": OUTPUT_DIMENSION
        }
    )

    embeddings = np.array([vector.values for vector in res.embeddings])
    normalized_vectors = embeddings / \
        np.linalg.norm(embeddings, axis=1, keepdims=True)

    return normalized_vectors


def get_top_k_2(feature_movie, k=10):
    return index.search(_get_embed(feature_movie), k)


def get_rcm(movies, k=10):
    recommend_contents = "\n======\n".join(
        movies["page_content"].tolist())

    user_desc = get_user_desc(recommend_contents)
    D, I = get_top_k_2(user_desc, k)

    return movie_df.iloc[I.flatten()]


def get_prefs(user_, K=50, threshold=3):
    high_rated = user_[user_["rating"] >= threshold]
    low_rated = user_[user_["rating"] < threshold]
    return user_.loc[high_rated.index.tolist()[:K//2] + low_rated.index.tolist()[:K//2]]

In [ ]:
user_rates

In [ ]:
descs = []
for i in tqdm(range(1, 201)):

    user_ = user_rates.loc[i]
    user_ = user_.merge(movie_df, on="movieId")[
        ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]
    user_["page_content"] = user_.apply(convert_prompt_2, axis=1)
    train, test = train_test_split(user_, test_size=0.25, shuffle=False)
    user_prefs = get_prefs(train)

    big_str = "\n======\n".join(
        user_prefs["page_content"].tolist())
    user_desc = get_user_desc(big_str)
    time.sleep(3)

    descs.append(user_desc)

In [ ]:
len(descs)

In [ ]:
import pickle

with open("data/1800-descriptions.pkl", "wb") as f:
    pickle.dump(descs, f)

In [ ]:
index_2 = faiss.IndexFlatIP(int(OUTPUT_DIMENSION))
index_2.add()

In [ ]:
index_2.ntotal

In [ ]:
batch_size = 100
batch_count = 0
# user_size = user_rates.index.get_level_values(0).unique().shape[0]
user_size = len(descs)
num_batches = (user_size + batch_size - 1) // batch_size

index_2 = faiss.IndexFlatIP(int(OUTPUT_DIMENSION))

for i in tqdm(range(0, user_size, batch_size)):
    temp = [i.text for i in descs]
    batch = temp[i:i+batch_size]
    embed_vectors = _get_embed(batch)
    batch_count += 1
    if (batch_count % 100 == 0):
        print(f"Processed batch {batch_count}/{num_batches}")
    if batch_count % 1500 == 0:
        print("Rate limit reached. Waiting for delay...")
        time.sleep(60)
    index_2.add(embed_vectors)

In [ ]:
faiss.write_index(index_2, "../output_index/user_features_115")

In [ ]:
descs[0].text

In [ ]:
D, I = index.search(np.array([index_2.reconstruct(0)]), 5)

movie_df.iloc[I.flatten()]

In [ ]:
# user_size = user_rates.index.get_level_values(0).unique().shape[0]
user_size = index_2.ntotal
K = [10, 20, 50]

metrics_results_2 = {
    "HR": {k: [] for k in K},
    "Precision": {k: [] for k in K},
    "Recall": {k: [] for k in K},
    "NDCG": {k: [] for k in K},
}

counter = 0

for i in tqdm(range(1, user_size+1)):
    precision = []
    recall_list = []
    ndcg_list = []

    if (counter == 1000):
        break

    user_ = user_rates.loc[i]

    counter += 1
    if (counter % 100 == 0):
        print(f"User count: {counter}")

    user_ = user_.merge(movie_df, on="movieId")[
        ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]

    # print(user_)

    train, test = train_test_split(user_, test_size=0.25, shuffle=False)

    user_description_embed = np.array([index_2.reconstruct(i-1)])

    for k_ in K:
        D, I = index.search(user_description_embed, k_)

        recommend_movies = movie_df.iloc[I.flatten()]

        final_rcm_list = recommend_movies
        overlap = np.intersect1d(final_rcm_list.index, test.index.values).size
        hr_k = 1 if overlap > 0 else 0

        precisionk = (overlap / k_)
        precision.append(precisionk)

        recall_k = overlap / len(test.index.values)
        recall_list.append(recall_k)

        test_movies = test.index.values
        relevances = [
            1 if movie in test_movies else 0 for movie in final_rcm_list]
        dcg = np.sum([rel / np.log2(idx + 2)
                      for idx, rel in enumerate(relevances)])

        ideal_hits = min(len(test_movies), k_)
        idcg = np.sum([1 / np.log2(i + 2) for i in range(ideal_hits)])

        ndcg_k = dcg / idcg if idcg > 0 else 0
        ndcg_list.append(ndcg_k)

        metrics_results_2["Precision"][k_].append(precisionk)
        metrics_results_2["Recall"][k_].append(recall_k)
        metrics_results_2["NDCG"][k_].append(ndcg_k)
        metrics_results_2["HR"][k_].append(hr_k)

In [ ]:
final_results_2 = {metric: [np.mean(metrics_results_2[metric][k])
                            for k in K] for metric in metrics_results_2}
results_df_2 = pd.DataFrame(final_results_2, index=K).T

results_df_2

In [ ]:
high_rated = movie_list[movie_list["rating"] >= 3.0]
low_rated = movie_list[movie_list["rating"] < 3.0]

In [ ]:
k = 50

In [ ]:
temp = high_rated.index.tolist()[:k//2] + low_rated.index.tolist()[:k//2]

In [ ]:
res = get_rcm(movie_list.loc[temp])

In [ ]:
res.loc[541]["page_content"]